# L4d: Production-Planning Shortest Path

A production process with alternative routes is a weighted directed graph: each step is an edge, each edge carries a cost, and the plan is the least-cost route from start to completion. In this lab, we model one such process, predict the cheapest route by hand, compute it with the two algorithms from L4c, and test how an equipment discount changes the decision.

> __Learning Objectives:__
>
> By the end of this lab, you should be able to:
> * __Model a production process as a weighted directed graph:__ Read a process edge list into vertices, directed edges, and nonnegative step costs, and name the candidate routes from start to completion. Explain why nonnegative costs make Dijkstra's algorithm valid for this graph.
> * __Compute and check the least-cost route:__ Add up the step costs of each candidate route by hand, then confirm the prediction with Dijkstra's algorithm and Bellman–Ford. Reconstruct the route from the predecessor map that both algorithms return.
> * __Test how one cost change moves the decision:__ Discount one production step on a copy of the graph and recompute the route while the baseline model stays unchanged. Implement the break-even calculation that gives the price below which the discounted route is the better choice.

Let's get started!
___

## Setup, Data, and Prerequisites
First, we set up the computational environment by including the `Include.jl` file and loading any needed resources.

> The [`include(...)` command](https://docs.julialang.org/en/v1/base/base/#include) evaluates the contents of the input source file, `Include.jl`, in the notebook's global scope. The `Include.jl` file sets paths, loads required external packages, etc. For additional information on functions and types used in this material, see the [Julia programming language documentation](https://docs.julialang.org/en/v1/).

Let's set up our code environment:

In [ ]:
include(joinpath(@__DIR__, "Include.jl")); # include the Include.jl file

The setup loads [the `VLDataScienceMachineLearningPackage.jl` package](https://github.com/varnerlab/VLDataScienceMachineLearningPackage.jl), which provides the graph types and the shortest-path solver; see [the package documentation](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/) for the functions and types used here. It also loads [the `Test` standard library](https://docs.julialang.org/en/v1/stdlib/Test/) for the checks, [the `DataFrames.jl` package](https://dataframes.juliadata.org/stable/) and [the `PrettyTables.jl` package](https://ronisbr.github.io/PrettyTables.jl/stable/) for tables, and [the `Plots.jl` package](https://docs.juliaplots.org/stable/) for the route figures.

The setup also loads [the `L4dProductionPlanning` module](src/Compute.jl) from [`src/Compute.jl`](src/Compute.jl), the file you complete in Task 3. The module is reloaded every time the setup cell runs, so saved edits become available without restarting the kernel. Notebook calls stay qualified, for example [the `L4dProductionPlanning.route_cost(...)` function](src/Compute.jl), so they always refer to the most recently loaded version.

### Constants
The figures in Tasks 2 and 3 draw the graph on a fixed layout that mirrors the schematic in Task 1. The `node_coordinates::Matrix{Float64}` array holds one row per vertex with its (x, y) position, and the `start_vertex::Int64` and `finish_vertex::Int64` constants name the two ends of every route:

In [ ]:
# Layout for the route figures: one row per vertex, (x, y) position. It mirrors the schematic in Task 1.
node_coordinates = [
    10.0 10.0 ; # 1 start
    11.0 10.0 ; # 2
    11.0 11.0 ; # 3 upper route
    13.0 11.0 ; # 4 upper route
    13.0 10.0 ; # 5
    11.0  9.0 ; # 6 lower route
    12.0  9.0 ; # 7 lower route
    13.0  9.0 ; # 8 lower route
    14.0 10.0 ; # 9 completion
];
start_vertex = 1;  # every route starts here
finish_vertex = 9; # every route ends here

### Implementation
The route figures in Tasks 2 and 3 share one drawing function. The `plotroute(...)` function draws every edge in gray with its cost, redraws the edges of one route in red, and marks the start vertex in green and the finish vertex in red to match the schematic:

In [ ]:
"""
    plotroute(graphmodel::MySimpleDirectedGraphModel, route::Vector{Int64}, coordinates::Matrix{Float64};
        start::Int64 = start_vertex, finish::Int64 = finish_vertex)

Draw the graph on the given layout, label every edge with its cost, and highlight `route` in red.
Returns the current figure.
"""
function plotroute(graphmodel::MySimpleDirectedGraphModel, route::Vector{Int64}, coordinates::Matrix{Float64};
    start::Int64 = start_vertex, finish::Int64 = finish_vertex)

    route_edges = Set((route[i], route[i + 1]) for i in 1:(length(route) - 1));
    plot(); # start a fresh figure

    # every edge in gray, labeled with its cost; route edges are labeled again in red afterward
    for ((s, t), w) in graphmodel.edges
        xs = [coordinates[s, 1], coordinates[t, 1]];
        ys = [coordinates[s, 2], coordinates[t, 2]];
        plot!(xs, ys, arrow = true, color = :gray90, lw = 2, label = "")
        if !((s, t) in route_edges)
            annotate!(sum(xs) / 2, sum(ys) / 2 + 0.15, text(string(round(w, digits = 2)), 8, :black))
        end
    end

    # the route in red, drawn on top of the gray edges
    for (s, t) in route_edges
        xs = [coordinates[s, 1], coordinates[t, 1]];
        ys = [coordinates[s, 2], coordinates[t, 2]];
        plot!(xs, ys, arrow = true, color = :red, lw = 2, label = "")
        annotate!(sum(xs) / 2, sum(ys) / 2 + 0.20, text(string(round(graphmodel.edges[(s, t)], digits = 2)), 8, :red))
    end

    # vertices: gray by default, green start, red finish, numbered
    scatter!(coordinates[:, 1], coordinates[:, 2], c = :gray, ms = 16, label = "")
    scatter!([coordinates[start, 1]], [coordinates[start, 2]], c = :green, ms = 16,
        markerstrokewidth = 2, markerstrokecolor = :darkgreen, label = "Start")
    scatter!([coordinates[finish, 1]], [coordinates[finish, 2]], c = :red, ms = 16,
        markerstrokewidth = 2, markerstrokecolor = :darkred, label = "Finish")
    for i in 1:size(coordinates, 1)
        color = (i == start || i == finish) ? :white : :black
        annotate!(coordinates[i, 1], coordinates[i, 2], text(string(i), 9, color))
    end

    plot!(axis = nothing, border = :none, legend = :outertopright, legendfontsize = 8,
        background_color = :white, xlim = (9.5, 14.5), ylim = (8.5, 11.5))
    return current()
end;

___

## Task 1: Build the production graph
Before we can compute anything, the process has to exist as a graph the solver understands. In this task, we read the process edge list into a directed graph model and inspect how the model stores its edges and costs.

The process has a start vertex, a completion vertex, and two ways to get between them: an upper route with fewer but more expensive steps, and a lower route with more steps that are each cheap. The schematic shows the vertices and the direction of every step.

<div>
    <center>
        <img src="figs/Fig-Branch-Schematic.svg" width="480" alt="Directed production graph: start vertex 1 leads to vertex 2, which splits into an upper route through vertices 3 and 4 and a lower route through vertices 6, 7, and 8; both routes rejoin at vertex 5 before the completion vertex 9"/>
    </center>
</div>

The costs live in [`data/Production-Process.edgelist`](data/Production-Process.edgelist), one record per step.

> __Records:__
>
> Each record has three comma-separated fields: `source`, `target`, and `cost`. The `source` and `target` fields are the integer ids of the vertices the step connects, and the `cost` field is the cost of completing that step, in arbitrary cost units. Lines that start with `#` are comments and are skipped.

The package reads the file for us and calls a parser we supply once per record. Our parser splits the record on the delimiter, converts the three fields, and raises an error on a record that does not have three fields:

In [ ]:
"""
    edgerecordparser(record::String, delim::Char = ',') -> Tuple{Int64, Int64, Float64}

Parse one `source,target,cost` record into the tuple the package expects.

### Arguments
- `record`: The edge record string to parse.
- `delim`: The delimiter used to split the record.

### Returns
- A tuple containing the source vertex id, target vertex id, and cost of the edge.

### Errors
- `ArgumentError`: The record does not have exactly three fields.
"""
function edgerecordparser(record::String, delim::Char = ',')

    fields = strip.(split(record, delim)); # remove whitespace around the fields
    length(fields) == 3 || throw(ArgumentError("expected source,target,cost but got: $(record)"))

    source = parse(Int64, fields[1]);  # source vertex id
    target = parse(Int64, fields[2]);  # target vertex id
    cost = parse(Float64, fields[3]);  # cost of completing the step

    return (source, target, cost)
end;

Next, let's set the path to the edge list file in the `path_to_edge_file::String` variable:

In [ ]:
path_to_edge_file = joinpath(CHEME5800_L4D_DATA, "Production-Process.edgelist"); # the graph shown in the schematic

Now we read the file into a dictionary [of `MyGraphEdgeModel` instances](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MyGraphEdgeModel), one per step, stored in the `myedgemodels::Dict{Int64, MyGraphEdgeModel}` variable. The keys are edge ids that count the data records in file order starting at zero, so the first step `(1, 2)` is edge `0`:

In [ ]:
myedgemodels = MyGraphEdgeModels(path_to_edge_file, edgerecordparser, delim = ',', comment = '#')

Since this is a directed graph, we build [a `MySimpleDirectedGraphModel` instance](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySimpleDirectedGraphModel) from the edge models using [the `build(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/factory/#VLDataScienceMachineLearningPackage.build). We store the graph in the `directedgraphmodel::MySimpleDirectedGraphModel` variable:

In [ ]:
directedgraphmodel = build(MySimpleDirectedGraphModel, myedgemodels);

What is inside the graph model? Fields belong to a type, not to an instance, so we ask the instance for its type [using the `typeof(...)` function](https://docs.julialang.org/en/v1/base/base/#Core.typeof) and then ask that type for its field names [using the `fieldnames(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.fieldnames):

In [ ]:
typeof(directedgraphmodel) |> T -> fieldnames(T) # the type T, then the field names of T

The `edgesinverse::Dict{Int64, Tuple{Int64, Int64}}` field maps an edge id to its `(source, target)` pair. The builder numbers the edges again, starting at one and ordered by source vertex and then target vertex, so these ids differ from the file record ids: the step `(3, 4)` is edge `2` in the file records and edge `4` in the graph model. We use the graph model's ids from here on:

In [ ]:
directedgraphmodel.edgesinverse

The `edges::Dict{Tuple{Int64, Int64}, Number}` field maps each `(source, target)` pair to its cost. Every cost in this process is nonnegative, which is the condition Dijkstra's algorithm requires:

In [ ]:
directedgraphmodel.edges

Let's collect the edges, their endpoints, and their costs in one table, ordered by the graph model's edge id:

In [ ]:
let
    edges = directedgraphmodel.edges;
    edgesinverse = directedgraphmodel.edgesinverse;
    df = DataFrame();
    for i in sort(collect(keys(edgesinverse)))
        (s, t) = edgesinverse[i];
        push!(df, (edge = i, s = s, t = t, cost = edges[(s, t)]));
    end
    pretty_table(df)
end

The upper route uses edges 2, 4, and 5, with costs 4, 8, and 2. The lower route uses edges 3, 7, 8, and 9, each with a cost of 2. Both routes share the first step, edge 1, and the last step, edge 6.
___

## Task 2: Compute the least-cost route
The graph is small enough to price each candidate route by hand, which gives us a prediction to check the solver against. In this task, we add up the step costs of the two routes, then confirm the prediction with Dijkstra's algorithm and Bellman–Ford.

The cost of a route is the sum of the costs of its steps. For a route $\langle v_0, v_1, \ldots, v_k\rangle$ with step costs $w(v_i, v_{i+1})$, the route cost is:
$$
C = \sum_{i=0}^{k-1} w(v_i, v_{i+1}).
$$

From the edge table, the upper route costs $1 + 4 + 8 + 2 + 1 = 16$ and the lower route costs $1 + 2 + 2 + 2 + 2 + 1 = 10$. So we predict that the lower route is the shortest path even though it has one more step.

[The `L4dProductionPlanning.route_cost(...)` function](src/Compute.jl) does the same sum for any route, and is already complete: it takes the `edges` dictionary and a vector of vertex ids, and raises an error for an empty route or a pair of vertices that is not an edge. The `upper_route::Vector{Int64}` and `lower_route::Vector{Int64}` variables hold the vertex ids of the two routes, and the `upper_cost::Float64` and `lower_cost::Float64` variables hold their computed costs:

In [ ]:
upper_route = [1, 2, 3, 4, 5, 9];    # fewer steps, two of them expensive
lower_route = [1, 2, 6, 7, 8, 5, 9]; # more steps, each of them cheap
upper_cost = L4dProductionPlanning.route_cost(directedgraphmodel.edges, upper_route);
lower_cost = L4dProductionPlanning.route_cost(directedgraphmodel.edges, lower_route);
(upper = upper_cost, lower = lower_cost)

The function agrees with the arithmetic. Dijkstra's algorithm should now return a distance of 10 to vertex 9 and a predecessor chain that traces the lower route.

To compute the shortest paths, we [call the `findshortestpath(...)` function](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/graphs/#VLDataScienceMachineLearningPackage.findshortestpath) with the graph model, the node model of the start vertex, and the algorithm to use. It returns two dictionaries: `d::Dict{Int64, Float64}` holds the distance from the start to every vertex, and `p::Dict{Int64, Union{Nothing, Int64}}` holds each vertex's predecessor on its shortest path. The start vertex has no predecessor, so following predecessors backward from any vertex ends at the start:

In [ ]:
(d, p) = let
    startnode = directedgraphmodel.nodes[start_vertex]; # the node model, not the id
    (d, p) = findshortestpath(directedgraphmodel, startnode, algorithm = DijkstraAlgorithm());
    (d, p)
end;

The route itself comes from [the `L4dProductionPlanning.reconstruct_route(...)` function](src/Compute.jl), which is already complete: it follows predecessors backward from the target until it reaches the start, then reverses the list. We store the result in the `shortest_route::Vector{Int64}` variable:

In [ ]:
shortest_route = L4dProductionPlanning.reconstruct_route(p, finish_vertex);
(distance = d[finish_vertex], route = shortest_route)

Do we get the lower route with a cost of 10? The figure highlights the computed route in red and labels every edge with its cost:

In [ ]:
plotroute(directedgraphmodel, shortest_route, node_coordinates)

Bellman–Ford reaches its answer by repeated relaxation of every edge rather than by Dijkstra's greedy choice, so it gives an independent check of the result. The call is the same; only the algorithm argument changes. We store the results in the `d_bellman::Dict{Int64, Float64}` and `p_bellman::Dict{Int64, Union{Nothing, Int64}}` variables:

In [ ]:
(d_bellman, p_bellman) = let
    startnode = directedgraphmodel.nodes[start_vertex];
    (d, p) = findshortestpath(directedgraphmodel, startnode, algorithm = BellmanFordAlgorithm());
    (d, p)
end;

The test set for this task checks the route-cost function against the hand arithmetic, the Dijkstra result against the prediction, and the Bellman–Ford result against Dijkstra:

In [ ]:
@testset "L4d least-cost route" begin
    # the route-cost function matches the hand arithmetic
    @test upper_cost == 16.0
    @test lower_cost == 10.0
    @test L4dProductionPlanning.route_cost(directedgraphmodel.edges, [start_vertex]) == 0.0
    @test_throws ArgumentError L4dProductionPlanning.route_cost(directedgraphmodel.edges, Int64[])
    @test_throws ArgumentError L4dProductionPlanning.route_cost(directedgraphmodel.edges, [1, 3])

    # Dijkstra agrees with the hand calculation
    @test d[finish_vertex] == lower_cost
    @test shortest_route == lower_route

    # Bellman–Ford agrees with Dijkstra
    @test d_bellman[finish_vertex] == d[finish_vertex]
    @test L4dProductionPlanning.reconstruct_route(p_bellman, finish_vertex) == shortest_route
end;

Dijkstra's algorithm returns the lower route at a cost of 10, the hand arithmetic gives the same number, and Bellman–Ford agrees on both the distance and the route. The upper route loses because its two expensive steps, at 4 and 8 cost units, outweigh its shorter length.
___

## Task 3: There was a sale on equipment!
The lower route won by 6 cost units, and the whole gap comes from the two expensive steps on the upper route. In this task, we discount the most expensive step, edge `(3, 4)` with a cost of 8, and ask whether the plan should change.

We change the cost on a copy of the graph so the baseline model stays intact for comparison. [The `deepcopy(...)` function](https://docs.julialang.org/en/v1/base/base/#Base.deepcopy) copies the model and every dictionary inside it, so the copy can be edited without touching the original. We store the copy in the `discounted_graphmodel::MySimpleDirectedGraphModel` variable:

In [ ]:
discounted_graphmodel = deepcopy(directedgraphmodel); # a copy we can edit; the baseline stays as it was

The [`MySimpleDirectedGraphModel` type is mutable](https://varnerlab.github.io/VLDataScienceMachineLearningPackage.jl/dev/types/#VLDataScienceMachineLearningPackage.MySimpleDirectedGraphModel), and its `edges` field is a dictionary, so we can assign a new cost to one key. The `discount_factor` value is the fraction of the original price that remains; a value of `0.0` means the equipment was donated and the step is free. Let's apply the discount to the copy:

In [ ]:
let
    discount_factor = 0.0;           # fraction of the original price that remains: 0.0 is free, 1.0 is no discount
    discounted_steps = [(3, 4)];     # the (source, target) pairs that go on sale
    for step in discounted_steps
        discounted_graphmodel.edges[step] *= discount_factor;
    end
end;

What are the new prices? The table compares the baseline and discounted cost of every edge:

In [ ]:
let
    edges = directedgraphmodel.edges;
    discounted_edges = discounted_graphmodel.edges;
    edgesinverse = directedgraphmodel.edgesinverse;
    df = DataFrame();
    for i in sort(collect(keys(edgesinverse)))
        (s, t) = edgesinverse[i];
        push!(df, (edge = i, s = s, t = t, cost = edges[(s, t)],
            discounted_cost = discounted_edges[(s, t)], Δ = discounted_edges[(s, t)] - edges[(s, t)]));
    end
    pretty_table(df)
end

Now we recompute the shortest path on the discounted graph. The results go in the `d₁::Dict{Int64, Float64}` and `p₁::Dict{Int64, Union{Nothing, Int64}}` variables, and the route in the `discounted_route::Vector{Int64}` variable:

In [ ]:
(d₁, p₁, discounted_route) = let
    startnode = discounted_graphmodel.nodes[start_vertex];
    (d, p) = findshortestpath(discounted_graphmodel, startnode, algorithm = DijkstraAlgorithm());
    (d, p, L4dProductionPlanning.reconstruct_route(p, finish_vertex))
end;

Does the discount change the plan? The figure highlights the new route:

In [ ]:
plotroute(discounted_graphmodel, discounted_route, node_coordinates)

With the step free, the upper route costs 8 and the lower route still costs 10, so the plan moves to the upper route. The baseline model is untouched: edge `(3, 4)` in `directedgraphmodel` still costs 8.

A free step is the extreme case. The practical question is how large the discount has to be before the plan changes.

Suppose the discounted step keeps a cost of $w$ and every other cost stays fixed. The upper route then costs $C_{\text{upper}} - w_{34} + w$, where $C_{\text{upper}}$ is its baseline cost and $w_{34}$ is the baseline cost of the step. The lower route does not use the step, so its cost $C_{\text{lower}}$ does not change. The two routes tie at the break-even cost:
$$
w^{\star} = C_{\text{lower}} - \left(C_{\text{upper}} - w_{34}\right).
$$

The upper route is strictly cheaper for any $w < w^{\star}$. The formula assumes the step appears once on the candidate route and not at all on the reference route; otherwise the discount would move both costs.

This calculation is the function you complete. It lives in [the `L4dProductionPlanning` module](src/Compute.jl) in [`src/Compute.jl`](src/Compute.jl).

> __Break-even contract__
>
> __Inputs__
>
> * `edges::AbstractDict`: the baseline `(source, target) => cost` dictionary.
> * `candidate_route::AbstractVector{<:Integer}`: the route that contains the discounted step exactly once.
> * `reference_route::AbstractVector{<:Integer}`: the route it competes against, which must not contain the step.
> * `step::Tuple{<:Integer, <:Integer}`: the `(source, target)` pair of the discounted step.
>
> __Output__
>
> * `Float64`: the cost of `step` at which the two routes cost the same. A negative value means no nonnegative price makes the candidate route cheaper.
>
> __Errors__
>
> * [`ArgumentError`](https://docs.julialang.org/en/v1/base/base/#Core.ArgumentError): the step does not appear exactly once on the candidate route, the step appears on the reference route, or either route fails the route-cost checks.

Complete the two TODOs in [the `breakeven_weight(...)` function](src/Compute.jl), save the file, and re-run the setup cell:

1. Check that the step appears exactly once on the candidate route and never on the reference route, raising an error otherwise.
2. Compute both route costs with the route-cost function and return the break-even cost from the formula.

The test set at the end of this task is the definition of done. We store the result in the `breakeven_cost::Float64` variable:

In [ ]:
breakeven_cost = L4dProductionPlanning.breakeven_weight(directedgraphmodel.edges, upper_route, lower_route, (3, 4))

The break-even cost should be 2, a discount of 75 percent from the baseline cost of 8. Let's check it on two more copies of the baseline graph, one with the step priced just below the break-even cost and one just above it:

In [ ]:
let
    df = DataFrame();
    for price in (breakeven_cost - 0.5, breakeven_cost + 0.5)
        trial = deepcopy(directedgraphmodel);
        trial.edges[(3, 4)] = price;
        (d, p) = findshortestpath(trial, trial.nodes[start_vertex], algorithm = DijkstraAlgorithm());
        route = L4dProductionPlanning.reconstruct_route(p, finish_vertex);
        push!(df, (price = price, distance = d[finish_vertex], route = join(route, " → ")));
    end
    pretty_table(df)
end

The final test set checks the discount scenario, the break-even cost, and that the baseline model was never changed:

In [ ]:
@testset "L4d discount scenario" begin
    # the copy changed and the baseline did not
    @test discounted_graphmodel.edges[(3, 4)] == 0.0
    @test directedgraphmodel.edges[(3, 4)] == 8.0

    # the plan moves to the upper route
    @test d₁[finish_vertex] == 8.0
    @test discounted_route == upper_route

    # the break-even contract
    @test breakeven_cost == 2.0
    @test_throws ArgumentError L4dProductionPlanning.breakeven_weight(directedgraphmodel.edges, upper_route, lower_route, (6, 7))
    @test_throws ArgumentError L4dProductionPlanning.breakeven_weight(directedgraphmodel.edges, upper_route, lower_route, (1, 2))
end;

Any nonnegative price below 2 for the discounted step moves the plan to the upper route, and any price above 2 leaves the lower route in place. The plan is sensitive to a single cost, which is why the inputs to a shortest-path model have to be stated as explicitly as the route it returns.
___

## Summary
In this lab, we modeled a production process as a weighted directed graph, predicted and computed its least-cost route with two algorithms, and showed that discounting one step by more than 75 percent moves the plan to the other route.

> __Key Takeaways:__
>
> * **A production plan is a shortest-path problem:** Steps become directed edges with nonnegative costs, and a plan is a route from start to completion. Because the costs are nonnegative, Dijkstra's algorithm is valid, and Bellman–Ford returns the same answer as an independent check.
> * **A route answer has three parts:** A shortest-path result is the route, its cost, and the predecessor map the route was rebuilt from. Pricing the candidate routes by hand first gave a prediction that the solver then confirmed.
> * **One cost can change the decision:** Editing a copy of the graph kept the baseline for comparison, and the break-even cost of 2 marked the price below which the upper route takes over. A shortest-path result is only as reliable as the costs it was computed from.

Next week we keep the graph but change the question: instead of the cheapest route through a network, we ask how much can flow through it, which leads to maximum-flow problems and linear programming.
___